# 02 · NLP Enrichment
**MLS NLP Pipeline — Stage 2**

Transforms raw articles into structured data by extracting:

| Component | Method | Output |
|-----------|--------|--------|
| Club mentions | Custom regex aliases (28 clubs) | `clubs_mentioned` |
| Person mentions | spaCy `en_core_web_sm` NER | `players/coaches/executives_mentioned` |
| Event type | Keyword pattern matching (10 categories) | `event_types`, `primary_event_type` |
| Sentiment | VADER compound score | `sentiment_compound`, `sentiment_label` |
| Temporal context | Month → season phase mapping | `season_phase`, `transfer_window` |

## 1. Setup

In [ ]:
import sys, re
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from pipeline.utils import get_config, load_parquet, get_parquet_path, PROJECT_ROOT
from pipeline.enricher import EntityMatcher, EventClassifier, SentimentAnalyzer, TemporalContexter

settings = get_config("settings")
entities = get_config("entities")
data_dir = PROJECT_ROOT / settings["pipeline"]["data_dir"]

print("spaCy model:", settings["enrichment"]["spacy_model"])
print("Entity match threshold:", settings["enrichment"]["entity_match_threshold"])

## 2. Club Entity Matching

In [ ]:
matcher = EntityMatcher(entities)

# Show alias patterns for a few clubs
sample_clubs = ["Inter Miami CF", "Seattle Sounders FC", "LAFC", "CF Montreal"]
for club in sample_clubs:
    cfg = next((c for c in entities["clubs"] if c["canonical"] == club), None)
    if cfg:
        print(f"{club}")
        print(f"  aliases : {cfg.get('aliases', [])}")
        print(f"  abbrevs : {cfg.get('abbreviations', [])}")
        print()

In [ ]:
# Live demo — extract clubs from a headline
test_texts = [
    "Lionel Messi scores on MLS debut for Inter Miami in Leagues Cup thriller",
    "Seattle Sounders defeat LAFC in Western Conference Final",
    "Toronto FC sign former Newcastle United midfielder ahead of 2024 season",
]
for text in test_texts:
    clubs = matcher.extract_clubs(text)
    print(f"TEXT  : {text[:80]}")
    print(f"CLUBS : {clubs}")
    print()

## 3. Person Extraction & Role Classification

In [ ]:
# spaCy NER with tight context window for role disambiguation
test_article = """
Head coach Steve Cherundolo guided LAFC to the MLS Cup title.
Carlos Vela scored the decisive goal in the final against Philadelphia Union.
Sporting director John Thorrington praised the squad depth.
"""

persons = matcher.extract_persons(test_article)
print("Extracted persons:")
for name, role in persons.items():
    print(f"  {name:<25} → {role}")

### How Role Classification Works

For each detected PERSON entity, we look at a **tight 25-char window before + 35-char window after**
the name for role keywords:

- **Coach keywords**: `head coach`, `assistant coach`, `technical director`, `manager`, ...
- **Executive keywords**: `president`, `CEO`, `sporting director`, `co-owner`, ...
- **Default**: `player`

This tight window prevents false positives from distant mentions in the same paragraph.

## 4. Event Classification

In [ ]:
import inspect
from pipeline.enricher import EventClassifier

clf = EventClassifier(entities)

headlines = [
    "Wayne Rooney fired as D.C. United head coach after poor start to season",
    "Inter Miami sign Sergio Busquets in blockbuster transfer deal",
    "Columbus Crew defeat New England Revolution 2-1 in MLS Cup final",
    "Atlanta United reveal $500M stadium renovation plan",
    "Clint Dempsey retires due to heart condition",
]

for h in headlines:
    types = clf.classify(h, "")
    print(f"  {types[0] if types else 'none':<20} | {h}")

## 5. Sentiment Analysis

In [ ]:
from pipeline.enricher import SentimentAnalyzer

analyzer = SentimentAnalyzer()

texts = [
    "Messi's debut was absolutely spectacular — the crowd was electric and euphoric",
    "The referee made a terrible decision that cost the team the match",
    "Columbus Crew win the MLS Cup after a hard-fought season",
    "Contract dispute leaves club in turmoil ahead of transfer window",
]

import pandas as pd
rows = []
for t in texts:
    score, label = analyzer.score(t)
    rows.append({"text": t[:60], "score": round(score, 3), "label": label})

pd.DataFrame(rows)

## 6. Running the Enricher — Sample Month

In [ ]:
# Load a raw month and enrich it
raw_path = get_parquet_path(data_dir / "press" / "raw", "raw", 2023, 7)
raw_df   = load_parquet(raw_path)
print(f"Raw articles: {len(raw_df)}")

enr_path = get_parquet_path(data_dir / "press" / "enriched", "enriched", 2023, 7)
enr_df   = load_parquet(enr_path)
print(f"Enriched articles: {len(enr_df)}")

enr_df[["title", "clubs_mentioned", "primary_event_type",
        "sentiment_compound", "season_phase"]].head(10)

## 7. Enrichment Quality Checks

In [ ]:
from pipeline.utils import load_all_parquet

enr_all = load_all_parquet(data_dir / "press" / "enriched")
print(f"Total enriched articles: {len(enr_all):,}")
print(f"Articles with club mentions : {(enr_all['clubs_mentioned'].str.len() > 0).sum():,} ({(enr_all['clubs_mentioned'].str.len() > 0).mean()*100:.1f}%)")
print(f"Articles with player mentions: {(enr_all['players_mentioned'].str.len() > 0).sum():,}")
print(f"Articles with coach mentions : {(enr_all['coaches_mentioned'].str.len() > 0).sum():,}")
print()
print("Event type distribution:")
print(enr_all["primary_event_type"].value_counts().to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sentiment distribution
enr_all["sentiment_compound"].hist(bins=40, ax=axes[0], color="#3498db", edgecolor="white")
axes[0].set_title("Sentiment Distribution")
axes[0].set_xlabel("VADER compound score")

# Event types
et = enr_all["primary_event_type"].value_counts().head(10)
et.plot(kind="barh", ax=axes[1], color="#e74c3c")
axes[1].set_title("Top Event Types")
axes[1].invert_yaxis()

# Season phase
sp = enr_all["season_phase"].value_counts()
sp.plot(kind="bar", ax=axes[2], color="#2ecc71", rot=30)
axes[2].set_title("Articles by Season Phase")

plt.tight_layout()
plt.show()